# Nica-GeoFetch: unidades hidrográficas Pfafstetter de Nicaragua

Este cuaderno público funciona al abrirse por sí solo en un entorno nuevo de Google Colab. Permite **diagnosticar, descargar, validar y convertir** las unidades hidrográficas nacionales de INETER (2025), niveles 4 a 7.

> Los datos institucionales son material de terceros y no están cubiertos por la licencia Apache-2.0 del software. No se ha identificado una licencia explícita de datos abiertos. Consulte a INETER antes de redistribuir copias completas.

El almacenamiento temporal de Colab es el destino predeterminado. Google Drive se monta únicamente después de una selección explícita.

## 1. Instalar Nica-GeoFetch

De forma predeterminada se instala desde `https://github.com/datanicaragua/nica-geofetch` usando la referencia configurable `GIT_REF`.

- Antes de la primera versión se usa `main`.
- Después de publicar versiones, cambie `GIT_REF` por una etiqueta estable, por ejemplo `v0.1.0`, para obtener resultados reproducibles.
- Si GitHub no está disponible, cambie `INSTALL_SOURCE` a `"zip"`; Colab solicitará un ZIP del paquete o repositorio para instalarlo manualmente.

In [ ]:
import subprocess
import sys

REPOSITORY_URL = "https://github.com/datanicaragua/nica-geofetch"
GIT_REF = "main"  # Antes de v0.1.0; luego use una etiqueta estable.
INSTALL_SOURCE = "github"  # Opciones: "github" o "zip".


def bootstrap_package():
    if INSTALL_SOURCE == "github":
        requirement = f"nica-geofetch[notebook] @ git+{REPOSITORY_URL}.git@{GIT_REF}"
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", requirement])
        print(f"Nica-GeoFetch instalado desde GitHub, referencia: {GIT_REF}")
        return
    if INSTALL_SOURCE == "zip":
        from google.colab import files

        print("Seleccione un ZIP del paquete o del repositorio Nica-GeoFetch.")
        uploaded = files.upload()
        candidates = [name for name in uploaded if name.lower().endswith((".zip", ".whl"))]
        if len(candidates) != 1:
            raise ValueError("Cargue exactamente un archivo .zip o .whl del paquete.")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", candidates[0]])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets>=8.1,<9"])
        print(f"Nica-GeoFetch instalado desde: {candidates[0]}")
        return
    raise ValueError('INSTALL_SOURCE debe ser "github" o "zip".')


bootstrap_package()

## 2. Seleccionar y ejecutar

Seleccione niveles y formato. **Diagnosticar acceso** hace una comprobación pequeña sobre un solo nivel. **Descargar y validar** procesa los niveles secuencialmente y genera un ZIP con datos, auditoría, procedencia y checksums. GeoPackage es el formato recomendado para análisis.

In [ ]:
import json
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display

from nica_geofetch.logging_utils import configure_logging
from nica_geofetch.models import OutputFormat
from nica_geofetch.providers.ineter_pfafstetter import IneterPfafstetterProvider
from nica_geofetch.workflows import download_workflow, import_local_workflow

configure_logging()
provider_selector = widgets.Dropdown(
    options=[("INETER Pfafstetter 2025", "ineter-pfafstetter")],
    description="Proveedor:",
)
level_checks = {
    level: widgets.Checkbox(value=(level == 4), description=f"Nivel {level}")
    for level in (4, 5, 6, 7)
}
format_selector = widgets.Dropdown(
    options=[
        ("KML original", "kml"),
        ("GeoPackage (recomendado)", "gpkg"),
        ("GeoJSON", "geojson"),
        ("Shapefile ZIP", "shapefile"),
        ("Todos", "all"),
    ],
    value="gpkg",
    description="Formato:",
)
location_selector = widgets.RadioButtons(
    options=[
        ("Temporal de Colab", "temporary"),
        ("Google Drive (montar explícitamente)", "drive"),
    ],
    value="temporary",
    description="Destino:",
)
diagnose_button = widgets.Button(description="Diagnosticar acceso", button_style="info")
download_button = widgets.Button(description="Descargar y validar", button_style="success")
progress = widgets.IntProgress(value=0, min=0, max=100, description="Progreso:")
log_output = widgets.Output()
LAST_RESULT = None


def selected_levels():
    levels = [level for level, control in level_checks.items() if control.value]
    if not levels:
        raise ValueError("Seleccione por lo menos un nivel.")
    return levels


def selected_formats():
    if format_selector.value == "all":
        return list(OutputFormat)
    return [OutputFormat(format_selector.value)]


def selected_output():
    if location_selector.value == "drive":
        from google.colab import drive

        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive/NicaGeoFetch_outputs")
    return Path("/content/NicaGeoFetch_outputs")


def diagnose_access(_button):
    with log_output:
        clear_output()
        level = selected_levels()[0]
        print(f"Diagnosticando el nivel {level} con una solicitud pequeña...")
        report = IneterPfafstetterProvider().diagnose(level)
        print(json.dumps(report.to_dict(), indent=2, ensure_ascii=False))


def download_and_validate(_button):
    global LAST_RESULT
    with log_output:
        clear_output()
        progress.value = 5
        levels = selected_levels()
        output = selected_output()
        print(f"Niveles: {levels}. Destino: {output}")
        print("Las solicitudes se harán de forma secuencial y respetuosa.")
        progress.value = 15
        LAST_RESULT = download_workflow(
            levels=levels,
            formats=selected_formats(),
            output_directory=output,
        )
        progress.value = 100
        print("Proceso completo. ZIP final:", LAST_RESULT.archive_path)
        display(LAST_RESULT.summary_rows())


diagnose_button.on_click(diagnose_access)
download_button.on_click(download_and_validate)
display(provider_selector)
display(widgets.HBox(list(level_checks.values())))
display(format_selector, location_selector)
display(widgets.HBox([diagnose_button, download_button]), progress, log_output)

## 3. Respaldo manual para el KML institucional

Si el acceso remoto falla, el diagnóstico muestra la URL oficial exacta. Ábrala en su navegador sin evadir controles institucionales, guarde el KML y cárguelo aquí. Indique el nivel correcto antes de importar.

In [ ]:
from google.colab import files

MANUAL_LEVEL = 4  # Cambie a 5, 6 o 7 según el archivo.
uploaded = files.upload()
if not uploaded:
    raise ValueError("No se cargó ningún archivo KML.")
manual_path = Path(next(iter(uploaded)))
manual_output = selected_output()
LAST_RESULT = import_local_workflow(
    input_path=manual_path,
    level=MANUAL_LEVEL,
    formats=selected_formats(),
    output_directory=manual_output,
)
display(LAST_RESULT.summary_rows())
print("ZIP final:", LAST_RESULT.archive_path)

## 4. Configuración simple si los widgets no funcionan

Edite las variables y ejecute la celda. Defina `MANUAL_KML` si ya descargó el archivo; déjela en `None` para usar el acceso oficial.

In [ ]:
LEVELS = [4]
FORMATS = ["gpkg"]  # kml, gpkg, geojson, shapefile o varios
OUTPUT_LOCATION = "temporary"  # temporary o drive
MANUAL_KML = None  # Por ejemplo: "/content/nivel4.kml"

if OUTPUT_LOCATION == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    simple_output = Path("/content/drive/MyDrive/NicaGeoFetch_outputs")
else:
    simple_output = Path("/content/NicaGeoFetch_outputs")

if MANUAL_KML:
    if len(LEVELS) != 1:
        raise ValueError("La importación manual procesa un nivel por archivo.")
    LAST_RESULT = import_local_workflow(
        input_path=Path(MANUAL_KML),
        level=LEVELS[0],
        formats=FORMATS,
        output_directory=simple_output,
    )
else:
    LAST_RESULT = download_workflow(
        levels=LEVELS,
        formats=FORMATS,
        output_directory=simple_output,
    )
display(LAST_RESULT.summary_rows())

## 5. Tabla final y descarga del ZIP

La tabla resume nivel, validez, cantidad de polígonos, checksum y formatos. El ZIP incluye archivos crudos y procesados, auditoría, manifiesto, procedencia, checksums y mapeos de campos.

In [ ]:
import pandas as pd
from google.colab import files

if LAST_RESULT is None:
    raise RuntimeError("Ejecute primero una descarga o importación manual.")
summary = pd.DataFrame(LAST_RESULT.summary_rows())
display(summary)
files.download(str(LAST_RESULT.archive_path))